In [1]:
import pandas as pd
import pandas as pd
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from uuid import uuid4

def load_players():
    players = pd.read_csv("data/nfl_draft_prospects.csv")
    college_stats = pd.read_csv("data/college_statistics.csv")
    profiles = pd.read_csv("data/nfl_draft_profiles.csv")

    return players, college_stats, profiles


PLAYERS, PLAYER_STATS, PROFILES = load_players()

In [2]:
PLAYERS.set_index("player_id", inplace=True)
PLAYER_STATS.set_index("player_id", inplace=True)
PROFILES.set_index("player_id", inplace=True)

# Pre-split by year
PLAYERS_BY_YEAR = {year: df for year, df in PLAYERS.groupby("draft_year")}

DRAFTS = {}

In [3]:
def player_ids_by_year(year):
    return set(PLAYERS_BY_YEAR[year].index)


def draft_order_by_year(year):
    df = PLAYERS_BY_YEAR[year].dropna(subset=["overall"]).sort_values("overall")

    return [
        {
            "overall": int(row["overall"]),
            "round": int(row["round"]),
            "pick": int(row["pick"]),
            "team": row["team"],
            "player_id": None,
        }
        for player_id, row in df.iterrows()
    ]


def new_draft_state(year, team):
    return {
        "year": year,
        "user_team": team,
        "index": 0,
        "round": 1,
        "pick": 1,
        "available_players": player_ids_by_year(year),
        "draft_order": draft_order_by_year(year),
        "status": "simulating",  # simulating → waiting_for_user → complete
    }


def user_turn(draft):
    current = draft["draft_order"][draft["index"]]
    return current["team"] == draft["user_team"]


# -----------------------
# Core engine
# -----------------------


def update_status(draft):
    if draft["index"] >= len(draft["draft_order"]):
        draft["status"] = "complete"
        return

    if user_turn(draft):
        draft["status"] = "waiting_for_user"
    else:
        draft["status"] = "simulating"


def advance_metadata(draft):
    if draft["index"] >= len(draft["draft_order"]):
        return

    next_pick = draft["draft_order"][draft["index"]]
    draft["round"] = next_pick["round"]
    draft["pick"] = next_pick["pick"]


def draft_player(draft, player_id):
    draft["available_players"].remove(player_id)

    pick = draft["draft_order"][draft["index"]]
    pick["player_id"] = player_id

    draft["index"] += 1
    advance_metadata(draft)
    update_status(draft)


def simulate_pick(draft):
    year_df = PLAYERS_BY_YEAR[draft["year"]]
    overall = draft["index"] + 1

    row = year_df[year_df["overall"] == overall]

    # Filter available players
    available = year_df[year_df.index.isin(draft["available_players"])]

    # If real pick exists and available → take it
    if len(row) > 0:
        pid = row.index[0]
        if pid in draft["available_players"]:
            draft_player(draft, pid)
            return

    # Otherwise choose best available
    if len(available) == 0:
        update_status(draft)
        return

    position = row.iloc[0]["position"] if len(row) > 0 else None

    if position:
        same_position = available[available["position"] == position]
    else:
        same_position = pd.DataFrame()

    if len(same_position) > 0:
        best = same_position.sort_values("pos_rk").iloc[0]
    else:
        best = available.sort_values("ovr_rk").iloc[0]

    draft_player(draft, best.name)


# -----------------------
# API models
# -----------------------


class StartDraftRequest(BaseModel):
    year: int
    user_team: str


# -----------------------
# API endpoints
# -----------------------

def get_years():
    return sorted(PLAYERS_BY_YEAR.keys())

def get_teams(year: int):
    if year not in PLAYERS_BY_YEAR:
        raise HTTPException(404, "Year not found")

    df = PLAYERS_BY_YEAR[year]
    return sorted(df["team"].dropna().unique().tolist())


def start_draft(req: StartDraftRequest):
    if req.year not in PLAYERS_BY_YEAR:
        raise HTTPException(404, "Year not found")

    draft_id = str(uuid4())
    DRAFTS[draft_id] = new_draft_state(req.year, req.user_team)
    return {"draft_id": draft_id}

def draft_status(draft_id: str):
    draft = DRAFTS.get(draft_id)
    if not draft:
        raise HTTPException(404, "Draft not found")

    return {
        "round": draft["round"],
        "pick": draft["pick"],
        "status": draft["status"],
        "index": draft["index"],
    }

def advance(draft_id: str):
    draft = DRAFTS.get(draft_id)
    if not draft:
        raise HTTPException(404, "Draft not found")

    while draft["status"] == "simulating":
        simulate_pick(draft)

    return {
        "round": draft["round"],
        "pick": draft["pick"],
        "status": draft["status"],
    }

def pick(draft_id: str, player_id: int):
    draft = DRAFTS.get(draft_id)
    if not draft:
        raise HTTPException(404, "Draft not found")

    if draft["status"] != "waiting_for_user":
        raise HTTPException(400, "Not your turn")

    if player_id not in draft["available_players"]:
        raise HTTPException(400, "Player not available")

    draft_player(draft, player_id)

    # resume sim automatically
    while draft["status"] == "simulating":
        simulate_pick(draft)

    return {
        "round": draft["round"],
        "pick": draft["pick"],
        "status": draft["status"],
    }

def draft_board(draft_id: str):
    draft = DRAFTS.get(draft_id)
    if not draft:
        raise HTTPException(404, "Draft not found")

    return {
        "year": draft["year"],
        "current_index": draft["index"],
        "board": draft["draft_order"],
        "status": draft["status"],
    }


def json_safe(val):
    return None if pd.isna(val) else val

def get_available_players(draft_id: str):
    draft = DRAFTS.get(draft_id)
    if not draft:
        raise HTTPException(status_code=404, detail="Draft not found")

    year = draft["year"]
    available_ids = draft["available_players"]

    df = PLAYERS_BY_YEAR.get(year)
    if df is None:
        raise HTTPException(status_code=404, detail="No players for this draft year")

    # Filter available players
    available_df = df[df.index.isin(available_ids)]

    # Sort by overall
    available_df = available_df.sort_values("overall")
    print(available_df[["height", "weight", "ovr_rk"]].head())
    # Return summary fields
    return [
        {
            "player_id": int(player_id),
            "name": row.player_name,
            "position": row.position,
            "height": json_safe(row.height),
            "weight": json_safe(row.weight),
            "overall_rank": None if pd.isna(row.ovr_rk) else int(row.ovr_rk),
        }
        for player_id, row in available_df.iterrows()
    ]

def get_player(player_id: int):
    if player_id not in PLAYERS.index:
        raise HTTPException(404, "Player not found")
    player = PLAYERS.loc[player_id]
    stats = (
        PLAYER_STATS.loc[player_id].to_dict() if player_id in PLAYER_STATS.index else {}
    )
    profile = PROFILES.loc[player_id].to_dict() if player_id in PROFILES.index else {}

    return {
        "player_id": int(player.name),
        "name": player.name,
        "position": player.position,
        "team": player.team,
        "height": player.height,
        "weight": player.weight,
        "college": player.college,
        "overall_rank": None if pd.isna(row.ovr_rk) else int(row.ovr_rk),
        "stats": stats,
        "profile": profile,
    }

In [4]:
req = StartDraftRequest(year=2001, user_team="Buffalo Bills")

draft_id = start_draft(req)['draft_id']

In [5]:
print(DRAFTS)
draft = DRAFTS.get(draft_id)
if not draft:
    raise HTTPException(status_code=404, detail="Draft not found")

year = draft["year"]
available_ids = draft["available_players"]

df = PLAYERS_BY_YEAR.get(year)
if df is None:
    raise HTTPException(status_code=404, detail="No players for this draft year")

# Filter available players
available_df = df[df.index.isin(available_ids)]

# Sort by overall
available_df = available_df.sort_values("overall")

{'73dc85df-ca5e-4e47-b953-e353324ce30e': {'year': 2001, 'user_team': 'Buffalo Bills', 'index': 0, 'round': 1, 'pick': 1, 'available_players': {1, 2, 3, 519, 520, 7, 12, 527, 528, 529, 530, 531, 20, 17, 22, 532, 24, 25, 26, 30, 32, 33, 37, 38, 46, 48, 50, 51, 59, 63, 64, 68, 70, 75, 76, 77, 79, 80, 81, 82, 84, 85, 597, 89, 90, 91, 93, 94, 95, 96, 99, 103, 104, 616, 107, 111, 113, 114, 116, 118, 119, 121, 123, 126, 131, 645, 136, 656, 145, 146, 147, 148, 150, 153, 154, 156, 157, 160, 161, 164, 165, 166, 169, 682, 175, 176, 177, 185, 189, 190, 191, 194, 197, 198, 201, 714, 204, 718, 719, 208, 209, 216, 217, 219, 221, 222, 223, 224, 226, 227, 228, 232, 234, 747, 235, 236, 237, 751, 240, 753, 754, 755, 244, 245, 246, 247, 252, 253, 254, 257, 264, 265, 266, 273, 275, 277, 279, 281, 283, 286, 287, 290, 295, 296, 297, 298, 301, 303, 307, 309, 310, 311, 313, 314, 317, 319, 321, 322, 326, 327, 330, 332, 337, 338, 340, 341, 346, 347, 352, 356, 357, 358, 359, 360, 363, 364, 365, 368, 369, 370, 371

In [6]:
available_df[["height", "weight", "ovr_rk"]]


,height,weight,ovr_rk
player_id,,,
25,NaN,NaN,NaN
145,NaN,NaN,NaN
378,NaN,NaN,NaN
368,NaN,NaN,NaN
520,NaN,NaN,NaN
...,...,...,...
147,NaN,NaN,NaN
597,NaN,NaN,NaN
402,NaN,NaN,NaN


In [7]:
available_df[["player_name", "position"]].isna().any()

player_name    False
position        True
dtype: bool